### 降采样
点更少，但仍能很好地代表环境，节省计算时间和难度  
也可以离群点去除？  
可以是每个voxel的中心点，也可以是每个voxel的平均点

### normal estimation法线估计
降采样后可以做法线估计  
法线估计可以用于后续的segmentation，或者是点云配准

In [7]:
import open3d as o3d
import numpy as np
import os
import sys

### 点云可视化

In [8]:
print("Load a ply point cloud, print it, and render it")
#把点云文件fragment.ply存入变量pcd中
pcd = o3d.io.read_point_cloud(r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\fragment.ply") 
print(pcd)
print(np.asarray(pcd.points))
o3d.visualization.draw_geometries(
    [pcd],
    zoom=0.3412, #控制画面的缩放程度
    front=[0.4257, -0.2125, -0.8795],  #表示虚拟相机观察点云的方向
    lookat=[2.6172, 2.0475, 1.532],  #虚拟相机正对的中心点
    up=[-0.0694, -0.9768, 0.2024] #规定画面中的上方是哪个方向
)
#后面这四个参数暂时没啥用，可视化点云用下面的语句就行
#o3d.visualization.draw_geometries([pcd])

Load a ply point cloud, print it, and render it
PointCloud with 196133 points.
[[0.65234375 0.84686458 2.37890625]
 [0.65234375 0.83984375 2.38430572]
 [0.66737998 0.83984375 2.37890625]
 ...
 [2.00839925 2.39453125 1.88671875]
 [2.00390625 2.39488506 1.88671875]
 [2.00390625 2.39453125 1.88793314]]


### Voxel downsampling 体素下采样
Voxel downsampling uses a regular voxel grid to create a uniformly downsampled point cloud from an input point cloud. It is often used as a pre-processing step for many point cloud processing tasks. The algorithm operates in two steps:
1. Points are bucketed into voxels.
2. Each occupied voxel generates exactly one point by averaging all points inside.

体素下采样使用规则的体素网格，将输入点云转换成一个均匀下采样后的点云，它经常作为许多点云处理任务的预处理步骤
该算法主要分为两个步骤：
1. 将点云中的点划分到不同的体素（voxel）中。
2. 每个包含点的体素最终只生成一个点，这个点的位置由该体素内所有点的位置取平均值得到。  

`pcd.voxel_down_sample(voxel_size = 0.05)`

In [ ]:
print("Downsample the point cloud with a voxel of 0.05")
#体素下采样，体素是边长为0.05的正方体
#voxel_size=0.05 本身没有固定单位，它的单位取决于你的点云坐标使用什么单位。如pcd的单位是m，那么体素边长就是0.05m
#voxel_size越大，点云下采样的程度越大，点云数量越少
downpcd =pcd.voxel_down_sample(voxel_size=0.05)   #将下采样后的点云保存到变量downdcpd中
o3d.visualization.draw_geometries([downpcd])  #可视化下采样后的点云

Downsample the point cloud with a voxel of 0.02


### Vertex normal estimation  顶点法向量估计
Another basic operation for point cloud is point normal estimation. Press `N`to see point normals. The keys`-` and `+` can be used to control the length of the normal.

点云的另一个基本操作是点法向量估计。按 `N`键可以查看点的法向量。使用`-` 和 `+`键可以控制法向量的显示长度。

`downpcd.estimate_nomals(search_param, radius, max_nn)`

In [12]:
print("recompute the normal of the downsampled point cloud")
downpcd.estimate_normals(   #给下采样后的downpcd的每个点计算一个法向量
    #计算点p的法向量时，Open3D会在点p周围寻找一批邻近点，然后根据这些邻近点估计出一个局部平面
    #垂直于这个局部平面的方向，就是点p的法向量
    #这里用的是hybrid混合邻域搜索，限制搜索半径radius为0.1，以及最大邻居点数max_nn为30
    search_param = o3d.geometry.KDTreeSearchParamHybrid(
        radius = 0.1,
        max_nn = 30
    )
)
o3d.visualization.draw_geometries([downpcd]) 

recompute the normal of the downsampled point cloud


### Access estimated vertex normal 访问每个估计的向量法线
Estimated normal vectors can be retrieved from the `normals` variable of `downpcd`

估计得到的法向量可以通过 `downpcd` 的 `normals` 变量获取。

In [ ]:
#查看降采样后的点云downpcd中第0个点的法向量
print("Print a normal vector of the 0th point")
print(downpcd.normals[0]) #类似于index

Print a normal vector of the 0th point
[-0.24558218 -0.15653157 -0.9566542 ]


To check out other variables, please use `help(downpcd)`. Normal vectors can be transformed as a numpy array using `np.asarray`  
如果想查看`downpcd`还有哪些其他变量，可以使用`help(downpcd)`。法向量可以使用`np.asarray`转换成 NumPy 数组。

In [14]:
print("Print the normal vectors of the first 10 points")
#np.asarray(downpcd.normals)把open3D的法向量转换为numpy数组，每一行是一个点的法向量的xyz坐标。有多少行就是有多少个点
print(np.asarray(downpcd.normals)[:10, :])
#[:10]表示取第 0～9 行，也就是前 10 个点。第二个冒号:表示所有列
#[1:8, 1:3] 表示取第 2～8 行，第 2～3 列
#[start:end]左闭右开，包前不包后

Print the normal vectors of the first 10 points
[[-2.45582183e-01 -1.56531570e-01 -9.56654200e-01]
 [ 7.49000381e-01  4.99041053e-02 -6.60687528e-01]
 [ 9.96759129e-01  3.16301276e-02 -7.39646773e-02]
 [ 9.13287245e-01 -2.27375588e-03 -4.07309757e-01]
 [-2.82147107e-01  6.67886298e-03 -9.59347905e-01]
 [ 6.83835183e-01 -6.92899388e-01 -2.28604200e-01]
 [ 7.00176032e-01  6.26365077e-04 -7.13969980e-01]
 [ 1.36463159e-03 -9.99401210e-01 -3.45739536e-02]
 [-5.85221626e-01  2.01289553e-01 -7.85492307e-01]
 [-2.12884156e-01 -2.66390544e-02 -9.76714235e-01]]
